# Notebook 5 — Grad-CAM Explainability
**RSNA Intracranial Hemorrhage Detection**

This notebook generates and validates Grad-CAM visualisations for the trained model.

### Contents
1. Load trained model (best_model.pth from Notebook 03)
2. Identify True Positives, False Positives, and False Negatives on validation set
3. Generate Grad-CAM heatmap overlays
4. **Occlusion sanity check** — verify Grad-CAM highlights causally important regions
5. Qualitative analysis: what the model attends to vs. clinical expectation

### Required inputs
- NB02 output: `manifest.csv` + `cache/` PNGs
- NB03 output: `best_model.pth`, `checkpoint.pth`

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
import os, gc, random, json as _json
import numpy as np
import pandas as pd
import cv2
from pathlib import Path

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Paths — update to match your committed notebook names ────────────────
CACHE_INPUT_DIR = '/kaggle/input/nb02eda'
NPY_CACHE_DIR   = f'{CACHE_INPUT_DIR}/cache'
MANIFEST_PATH   = f'{CACHE_INPUT_DIR}/manifest.csv'
MODEL_PATH      = '/kaggle/input/ich-train-session-final/best_model.pth'
CHECKPOINT      = '/kaggle/input/ich-train-session-final/checkpoint.pth'

ARCH         = 'efficientnet_b0'   # must match what was trained
IMG_SIZE     = 256
BATCH_SIZE   = 32
NUM_WORKERS  = 4
SEED         = 42

# ─── Load normalization stats ────────────────────────────────────────────
_norm_path = os.path.join(CACHE_INPUT_DIR, 'normalization_stats.json')
if os.path.exists(_norm_path):
    with open(_norm_path) as f:
        _norm = _json.load(f)
    MEAN = _norm['mean']
    STD  = _norm['std']
    print(f'Dataset normalization: mean={MEAN}, std={STD}')
else:
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]
    print(f'Using ImageNet defaults: mean={MEAN}, std={STD}')

random.seed(SEED)
np.random.seed(SEED)
print(f'Device: {DEVICE}')

In [ ]:
# ── 1. Load model ─────────────────────────────────────────────────────────
def build_model(arch: str) -> nn.Module:
    if arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=None)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 1))
    elif arch == 'resnet50':
        m = models.resnet50(weights=None)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 1))
    else:
        raise ValueError(arch)
    return m


model = build_model(ARCH)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE).eval()

# Load optimal threshold from checkpoint
ckpt = torch.load(CHECKPOINT, map_location='cpu')
THRESHOLD = ckpt.get('best_thresh', 0.5)
print(f'Model loaded. Optimal threshold: {THRESHOLD:.4f}')

In [ ]:
# ── 2. Validation dataset ─────────────────────────────────────────────────
class ICHDataset(Dataset):
    def __init__(self, df, npy_root, transform):
        self.df = df.reset_index(drop=True)
        self.npy_root = npy_root
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.npy_root, f'{row["image_id"]}.npy')
        try:
            img = np.load(path)                        # uint8 H×W×3 [0,255]
        except Exception:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        return self.transform(img), torch.tensor(float(row['any']), dtype=torch.float32), row['image_id']


val_transform = T.Compose([
    T.ToPILImage(), T.ToTensor(), T.Normalize(mean=MEAN, std=STD)
])

manifest = pd.read_csv(MANIFEST_PATH)
val_df   = manifest[manifest['split'] == 'val'].reset_index(drop=True)
val_ds   = ICHDataset(val_df, NPY_CACHE_DIR, val_transform)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
print(f'Validation set: {len(val_df):,} images')

In [ ]:
# ── 3. Run inference — collect predictions ────────────────────────────────
@torch.no_grad()
def run_inference(model, loader, threshold):
    model.eval()
    records = []
    for imgs, labels, ids in tqdm(loader, desc='Inference'):
        with torch.cuda.amp.autocast():
            logits = model(imgs.to(DEVICE)).squeeze(1).cpu().float()
        probs = torch.sigmoid(logits).numpy()
        preds = (probs >= threshold).astype(int)
        for img_id, lbl, prob, pred in zip(ids, labels.numpy(), probs, preds):
            records.append({
                'image_id': img_id,
                'label'   : int(lbl),
                'prob'    : round(float(prob), 4),
                'pred'    : int(pred),
            })
    return pd.DataFrame(records)


pred_df = run_inference(model, val_loader, THRESHOLD)

# Categorise each prediction
def categorise(row):
    if row['label'] == 1 and row['pred'] == 1: return 'TP'
    if row['label'] == 0 and row['pred'] == 0: return 'TN'
    if row['label'] == 1 and row['pred'] == 0: return 'FN'
    return 'FP'

pred_df['category'] = pred_df.apply(categorise, axis=1)

val_auc = roc_auc_score(pred_df['label'], pred_df['prob'])
print(f'Validation AUC: {val_auc:.5f}')
print(pred_df['category'].value_counts().to_string())

In [ ]:
# ── 4. Grad-CAM implementation ────────────────────────────────────────────
class GradCAM:
    """
    Hook-based Grad-CAM.
    Works with EfficientNet-B0 and ResNet-50.
    """

    def __init__(self, model: nn.Module, arch: str):
        self.model      = model
        self.gradients  = None
        self.activations = None

        # Pick the target layer (last spatial conv block)
        if arch == 'efficientnet_b0':
            target = model.features[-1]
        elif arch == 'resnet50':
            target = model.layer4[-1]
        else:
            raise ValueError(arch)

        self._fwd_hook = target.register_forward_hook(self._fwd_hook_fn)
        self._bwd_hook = target.register_full_backward_hook(self._bwd_hook_fn)

    def _fwd_hook_fn(self, module, input, output):
        self.activations = output.detach()

    def _bwd_hook_fn(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove(self):
        self._fwd_hook.remove()
        self._bwd_hook.remove()

    def generate(self, img_tensor: torch.Tensor) -> np.ndarray:
        """
        img_tensor: (1, C, H, W) normalised tensor on DEVICE.
        Returns: (H, W) heatmap in [0, 1].
        """
        self.model.zero_grad()
        img_tensor = img_tensor.clone().requires_grad_(True)

        logit = self.model(img_tensor).squeeze()     # scalar logit
        logit.backward()                             # differentiate w.r.t. target layer

        grads = self.gradients.squeeze()             # (C, H, W)
        acts  = self.activations.squeeze()           # (C, H, W)

        weights = grads.mean(dim=(1, 2), keepdim=True)   # global average pool over spatial dims
        cam     = (weights * acts).sum(dim=0)            # weighted sum over channels
        cam     = torch.relu(cam)                        # ReLU — keep only positive contributions
        cam     = cam.cpu().numpy()

        # Normalise to [0, 1]
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


grad_cam = GradCAM(model, ARCH)
print('Grad-CAM ready.')

In [ ]:
# ── 5. Overlay utility ────────────────────────────────────────────────────
def overlay_cam(original_rgb: np.ndarray, cam: np.ndarray,
                alpha: float = 0.45) -> np.ndarray:
    """
    Blend Grad-CAM heatmap over the original image.
    original_rgb: (H, W, 3) uint8
    cam         : (h, w) float [0,1]
    Returns: (H, W, 3) uint8 blended image.
    """
    H, W = original_rgb.shape[:2]
    cam_resized = cv2.resize(cam, (W, H), interpolation=cv2.INTER_LINEAR)
    heatmap     = (cm.jet(cam_resized)[:, :, :3] * 255).astype(np.uint8)   # (H,W,3)
    blended     = (alpha * heatmap + (1 - alpha) * original_rgb).astype(np.uint8)
    return blended


def load_npy(image_id: str) -> np.ndarray:
    """Load cached NPY and return as uint8 RGB for display/overlay."""
    path = os.path.join(NPY_CACHE_DIR, f'{image_id}.npy')
    try:
        return np.load(path)                           # uint8 [0,255]
    except Exception:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)


def tensor_from_id(image_id: str) -> torch.Tensor:
    img  = load_npy(image_id)
    t    = val_transform(img).unsqueeze(0).to(DEVICE)
    return t


print('Overlay utilities defined.')

In [ ]:
# ── 6. Generate Grad-CAM for TP / FN / FP cases ──────────────────────────
model.train()   # needed so gradients flow in EfficientNet (dropout, BN in eval can prevent grads)

N_EACH = 4      # number of examples per category to show

categories = [
    ('TP', 'True Positive  (correct detection)',   '#2ecc71'),
    ('FN', 'False Negative (missed hemorrhage)',   '#e74c3c'),
    ('FP', 'False Positive (false alarm)',         '#f39c12'),
]

for cat, cat_label, color in categories:
    subset = pred_df[pred_df['category'] == cat].sample(
        min(N_EACH, len(pred_df[pred_df['category'] == cat])), random_state=SEED
    )
    if len(subset) == 0:
        print(f'No {cat} cases found.'); continue

    fig, axes = plt.subplots(len(subset), 2, figsize=(8, len(subset) * 3.5))
    if len(subset) == 1:
        axes = axes[np.newaxis, :]   # keep 2D for consistent indexing

    fig.suptitle(f'Grad-CAM: {cat_label}', fontsize=12, color=color)

    for row_i, (_, row) in enumerate(subset.iterrows()):
        img_id = row['image_id']
        t      = tensor_from_id(img_id)
        cam    = grad_cam.generate(t)

        orig   = load_npy(img_id)
        blended = overlay_cam(orig, cam)

        prob_str = f'p={row["prob"]:.3f}'
        axes[row_i, 0].imshow(orig)
        axes[row_i, 0].set_title(f'{img_id[-8:]} | label={row["label"]} | {prob_str}', fontsize=8)
        axes[row_i, 0].axis('off')

        axes[row_i, 1].imshow(blended)
        axes[row_i, 1].set_title('Grad-CAM overlay', fontsize=8)
        axes[row_i, 1].axis('off')

    plt.tight_layout()
    save_path = f'/kaggle/working/gradcam_{cat.lower()}.png'
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

In [ ]:
# ── 7. Occlusion sanity check ─────────────────────────────────────────────
"""
For each high-confidence TP, we:
1. Locate the region of maximum Grad-CAM activation
2. Occlude it with a grey patch (mean pixel value)
3. Measure the drop in model confidence

If Grad-CAM is marking causally relevant regions, occluding them should
reduce the model's hemorrhage confidence noticeably.
"""

PATCH_SIZE = 48   # how many pixels to occlude per side

def occlude_at(img_tensor: torch.Tensor, cam: np.ndarray,
               patch_size: int = 48) -> torch.Tensor:
    """Replace the peak Grad-CAM region with zeros (grey after normalisation)."""
    cam_up = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    flat_idx = cam_up.argmax()
    cy, cx   = divmod(int(flat_idx), IMG_SIZE)

    y0 = max(cy - patch_size // 2, 0)
    y1 = min(cy + patch_size // 2, IMG_SIZE)
    x0 = max(cx - patch_size // 2, 0)
    x1 = min(cx + patch_size // 2, IMG_SIZE)

    t_occ = img_tensor.clone()
    t_occ[:, :, y0:y1, x0:x1] = 0.0   # zero = ImageNet mean after normalisation
    return t_occ, (y0, y1, x0, x1)


@torch.no_grad()
def get_prob(model, t):
    return torch.sigmoid(model(t).squeeze()).item()


model.eval()   # switch back to eval for inference

high_conf_tp = pred_df[
    (pred_df['category'] == 'TP') & (pred_df['prob'] > 0.75)
].head(8)

occlusion_records = []
fig, axes = plt.subplots(len(high_conf_tp), 3,
                         figsize=(12, len(high_conf_tp) * 3))
if len(high_conf_tp) == 1:
    axes = axes[np.newaxis, :]
fig.suptitle('Occlusion Sanity Check — High-confidence True Positives', fontsize=12)

for row_i, (_, row) in enumerate(high_conf_tp.iterrows()):
    img_id = row['image_id']
    t      = tensor_from_id(img_id)

    # Generate CAM (needs grad)
    model.train()
    cam = grad_cam.generate(t)
    model.eval()

    # Original probability
    orig_prob = get_prob(model, t)

    # Occlude and re-evaluate
    t_occ, bbox = occlude_at(t, cam, PATCH_SIZE)
    occ_prob    = get_prob(model, t_occ)
    drop        = orig_prob - occ_prob

    occlusion_records.append({
        'image_id': img_id,
        'orig_prob': round(orig_prob, 4),
        'occ_prob' : round(occ_prob, 4),
        'prob_drop': round(drop, 4),
    })

    orig_rgb = load_npy(img_id)
    cam_overlay = overlay_cam(orig_rgb, cam)

    # Draw occlusion patch on original
    occ_vis = orig_rgb.copy()
    y0, y1, x0, x1 = bbox
    occ_vis[y0:y1, x0:x1] = 127   # grey patch

    axes[row_i, 0].imshow(orig_rgb)
    axes[row_i, 0].set_title(f'Original  p={orig_prob:.3f}', fontsize=8)
    axes[row_i, 0].axis('off')

    axes[row_i, 1].imshow(cam_overlay)
    axes[row_i, 1].set_title('Grad-CAM', fontsize=8)
    axes[row_i, 1].axis('off')

    axes[row_i, 2].imshow(occ_vis)
    axes[row_i, 2].set_title(f'Occluded  p={occ_prob:.3f}  Δ={drop:+.3f}',
                              color='red' if drop > 0.05 else 'grey', fontsize=8)
    axes[row_i, 2].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/occlusion_sanity_check.png', bbox_inches='tight')
plt.show()

occ_df = pd.DataFrame(occlusion_records)
print(occ_df.to_string(index=False))
print(f'\nMean probability drop when occluding Grad-CAM peak: {occ_df["prob_drop"].mean():.4f}')
print('(A positive mean drop confirms Grad-CAM is highlighting causally relevant regions.)')

In [ ]:
# ── 8. Qualitative failure analysis (FN cases) ────────────────────────────
"""
For False Negatives (missed hemorrhages), we:
- Show the Grad-CAM overlay
- Note what the model attends to
- Compare with where a haemorrhage would clinically be expected
This documents typical failure patterns (small bleeds, artefacts, etc.)
"""
model.train()
fn_subset = pred_df[pred_df['category'] == 'FN'].sample(
    min(6, len(pred_df[pred_df['category'] == 'FN'])), random_state=SEED
)

fig, axes = plt.subplots(len(fn_subset), 2,
                         figsize=(8, len(fn_subset) * 3.5))
if len(fn_subset) == 1: axes = axes[np.newaxis, :]
fig.suptitle('False Negatives — Model attention (Grad-CAM)', fontsize=12, color='red')

fn_notes = []
for row_i, (_, row) in enumerate(fn_subset.iterrows()):
    img_id = row['image_id']
    t = tensor_from_id(img_id)
    cam = grad_cam.generate(t)

    orig_rgb = load_npy(img_id)
    blended  = overlay_cam(orig_rgb, cam)

    # Max attention coordinates
    cam_up  = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    flat_i  = cam_up.argmax()
    peak_y, peak_x = divmod(int(flat_i), IMG_SIZE)
    # Normalised peak location (0=left/top, 1=right/bottom)
    peak_nx = peak_x / IMG_SIZE
    peak_ny = peak_y / IMG_SIZE

    fn_notes.append({'image_id': img_id, 'prob': row['prob'],
                     'peak_x_norm': round(peak_nx, 2),
                     'peak_y_norm': round(peak_ny, 2)})

    axes[row_i, 0].imshow(orig_rgb)
    axes[row_i, 0].set_title(f'{img_id[-8:]}  p={row["prob"]:.3f} (FN)', fontsize=8)
    axes[row_i, 0].axis('off')

    axes[row_i, 1].imshow(blended)
    axes[row_i, 1].set_title(f'Grad-CAM  peak≈({peak_nx:.2f}, {peak_ny:.2f})', fontsize=8)
    axes[row_i, 1].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/gradcam_fn_analysis.png', bbox_inches='tight')
plt.show()

model.eval()   # restore eval mode

fn_df = pd.DataFrame(fn_notes)
fn_df.to_csv('/kaggle/working/fn_gradcam_notes.csv', index=False)
print(fn_df.to_string(index=False))
print()
print('Review observations:')
print(' - Do FN cases have lower contrast hemorrhage (small/subtle bleeds)?')
print(' - Does Grad-CAM highlight artifacts (skull edges, scan markers)?')
print(' - Are FN probabilities clustered just below the threshold?')
print(' These patterns inform target preprocessing or threshold adjustments.')

In [ ]:
# ── 9. Skull-edge attention analysis ──────────────────────────────────────
"""
If Grad-CAM consistently highlights skull edges or image borders rather than
brain parenchyma, the model may be using bone artifacts as a shortcut.
This cell quantifies how often the peak attention falls near the image border.

BORDER MARGIN: pixels within 15% of the edge are considered 'edge attention'.
"""
BORDER_MARGIN = 0.15   # fraction of image dimension

tp_subset = pred_df[pred_df['category'] == 'TP'].sample(
    min(50, len(pred_df[pred_df['category'] == 'TP'])), random_state=SEED
)

model.train()
edge_count = 0
total_checked = 0

for _, row in tp_subset.iterrows():
    t = tensor_from_id(row['image_id'])
    cam = grad_cam.generate(t)
    cam_up = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))

    flat_idx = cam_up.argmax()
    peak_y, peak_x = divmod(int(flat_idx), IMG_SIZE)
    norm_x = peak_x / IMG_SIZE
    norm_y = peak_y / IMG_SIZE

    # Check if peak is near any edge
    near_edge = (norm_x < BORDER_MARGIN or norm_x > (1 - BORDER_MARGIN) or
                 norm_y < BORDER_MARGIN or norm_y > (1 - BORDER_MARGIN))
    if near_edge:
        edge_count += 1
    total_checked += 1

model.eval()

edge_pct = edge_count / total_checked * 100 if total_checked > 0 else 0
print(f'Skull-edge attention analysis (TP cases):')
print(f'  Checked  : {total_checked} images')
print(f'  Edge peak: {edge_count} ({edge_pct:.1f}%)')
print(f'  Border margin: {BORDER_MARGIN*100:.0f}% of image edge')
print()
if edge_pct > 40:
    print('  ⚠ WARNING: >40% of TP cases have peak attention near skull/border.')
    print('  This may indicate the model is using bone artifacts as shortcuts.')
    print('  Consider: more aggressive cropping augmentation or skull-stripping.')
elif edge_pct > 20:
    print('  ⚡ NOTE: ~20-40% edge attention is common with brain CT.')
    print('  Monitor but not necessarily problematic.')
else:
    print('  ✓ Most attention falls on brain parenchyma — good sign.')

In [ ]:
# ── 10. Cleanup hooks ──────────────────────────────────────────────────────
grad_cam.remove()
print('Grad-CAM hooks removed.')
print('\nSaved outputs:')
print(' gradcam_tp.png       — TP overlay examples')
print(' gradcam_fn.png       — FN overlay examples')
print(' gradcam_fp.png       — FP overlay examples')
print(' occlusion_sanity_check.png  — sanity check results')
print(' gradcam_fn_analysis.png     — failure analysis')
print(' fn_gradcam_notes.csv        — FN attention coordinates')

In [ ]:
# ── HEALTH CHECK — automated output validation ────────────────────────────
import json as _json_hc

errors = []

expected_files = [
    'gradcam_tp.png', 'gradcam_fn.png', 'gradcam_fp.png',
    'occlusion_sanity_check.png', 'gradcam_fn_analysis.png',
    'fn_gradcam_notes.csv',
]
for f in expected_files:
    if not os.path.exists(f'/kaggle/working/{f}'):
        errors.append(f'Missing: {f}')

# Check occlusion results make sense
if len(occ_df) > 0:
    mean_drop = occ_df['prob_drop'].mean()
    if mean_drop < 0.01:
        errors.append(f'Occlusion mean drop={mean_drop:.4f} — Grad-CAM may not be highlighting causal regions')

health = {
    'notebook'   : '05_gradcam',
    'status'     : 'PASS' if not errors else 'FAIL',
    'errors'     : errors,
    'val_auc'    : round(float(val_auc), 5),
    'n_tp'       : int((pred_df['category'] == 'TP').sum()),
    'n_fn'       : int((pred_df['category'] == 'FN').sum()),
    'n_fp'       : int((pred_df['category'] == 'FP').sum()),
    'occ_mean_drop'  : round(float(occ_df['prob_drop'].mean()), 4) if len(occ_df) > 0 else None,
    'edge_attention_pct': round(edge_pct, 1),
}

with open('/kaggle/working/health_check_nb05.json', 'w') as f:
    _json_hc.dump(health, f, indent=2)

if errors:
    print('❌ HEALTH CHECK FAILED:')
    for e in errors:
        print(f'   • {e}')
else:
    print('✅ HEALTH CHECK PASSED')
    print(f'   Val AUC        : {val_auc:.5f}')
    print(f'   Occlusion drop : {occ_df["prob_drop"].mean():.4f} (mean)')
    print(f'   Edge attention  : {edge_pct:.1f}%')
    print(f'   Plots saved    : {len(expected_files)}')